In [9]:
%pip install numpy pandas scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path
from datetime import datetime
import json
import math

import numpy as np
import pandas as pd
import joblib

from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# Find your data folder.
DATA_DIR = Path("data")

if not DATA_DIR.is_dir():
    DATA_DIR = Path("../data")

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        "Cannot find the data folder. Set DATA_DIR to its full path, "
        "for example Path('C:/Users/YourName/Documents/data')."
    )

# A new results folder for each run.
OUTPUT_DIR = Path(
    "cow_results_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

# Compare short windows against your original 60-second duration.
WINDOW_SECONDS = [5, 10, 20, 60]

CHUNK_SIZE = 200_000
MATCH_TOLERANCE = pd.Timedelta("100ms")

SAMPLE_RATE = 10
MIN_SENSOR_COVERAGE = 0.80
MIN_LABEL_COVERAGE = 0.95
MAX_INTERNAL_GAP_SECONDS = 0.5

N_TREES = 200
N_JOBS = 2
RANDOM_STATE = 42

LABELS = [0, 1, 2]
CLASS_NAMES = ["Other", "Ruminating", "Eating_or_Drinking"]

# These columns must NEVER be used as predictors.
METADATA_COLUMNS = [
    "cow_id",
    "classification",
    "purity",
    "samples",
    "matched_samples",
]

print("Data folder:", DATA_DIR.resolve())
print("Results folder:", OUTPUT_DIR.resolve())

Data folder: C:\Users\firdaus.amyar\Downloads\SamsungSFT\backend\data
Results folder: C:\Users\firdaus.amyar\Downloads\SamsungSFT\backend\notebooks\cow_results_20260907_222727


In [11]:
def read_sensor_chunks(path, columns):
    previous_timestamp = None

    for frame in pd.read_csv(
        path,
        usecols=columns,
        chunksize=CHUNK_SIZE,
    ):
        if frame.empty:
            continue

        frame["timestamp"] = pd.to_datetime(
            frame["timestamp"],
            errors="raise",
        )

        timestamps = frame["timestamp"]

        if (
            timestamps.isna().any()
            or not timestamps.is_monotonic_increasing
            or timestamps.duplicated().any()
        ):
            raise ValueError(
                f"{path.name}: timestamps must be valid, sorted, and unique."
            )

        if (
            previous_timestamp is not None
            and timestamps.iloc[0] <= previous_timestamp
        ):
            raise ValueError(
                f"{path.name}: timestamps overlap or are out of order "
                "across file chunks."
            )

        previous_timestamp = timestamps.iloc[-1]

        for column in columns[1:]:
            frame[column] = pd.to_numeric(
                frame[column],
                errors="raise",
            )

        values = frame[columns[1:]].to_numpy(dtype=float)

        if not np.isfinite(values).all():
            raise ValueError(
                f"{path.name}: missing or non-finite sensor values/labels."
            )

        if (
            "classification" in frame.columns
            and not frame["classification"].isin(LABELS).all()
        ):
            raise ValueError(
                f"{path.name}: classification must contain only 0, 1, 2."
            )

        yield frame


def complete_time_blocks(reader, block_seconds):
    """Keep unfinished time blocks for the next chunk."""
    carry = None

    for chunk in reader:
        if carry is not None:
            frame = pd.concat([carry, chunk], ignore_index=True)
        else:
            frame = chunk

        boundary = frame["timestamp"].iloc[-1].floor(
            f"{block_seconds}s"
        )

        complete = frame.loc[frame["timestamp"] < boundary]
        carry = frame.loc[frame["timestamp"] >= boundary].copy()

        if not complete.empty:
            yield complete

    if carry is not None and not carry.empty:
        yield carry


class HalterStream:
    def __init__(self, reader):
        self.reader = iter(reader)
        self.buffer = None
        self.finished = False

    def match(self, acceleration):
        lower = acceleration["timestamp"].iloc[0] - MATCH_TOLERANCE
        upper = acceleration["timestamp"].iloc[-1] + MATCH_TOLERANCE

        if self.buffer is not None:
            self.buffer = self.buffer.loc[
                self.buffer["timestamp"] >= lower
            ].copy()

        while not self.finished and (
            self.buffer is None
            or self.buffer.empty
            or self.buffer["timestamp"].iloc[-1] <= upper
        ):
            try:
                next_chunk = next(self.reader)
            except StopIteration:
                self.finished = True
                break

            next_chunk = next_chunk.loc[
                next_chunk["timestamp"] >= lower
            ]

            if self.buffer is None:
                self.buffer = next_chunk.copy()
            else:
                self.buffer = pd.concat(
                    [self.buffer, next_chunk],
                    ignore_index=True,
                )

        if self.buffer is None or self.buffer.empty:
            result = acceleration.copy()
            result["halter_timestamp"] = pd.NaT
            result["classification"] = np.nan
            return result

        reference = self.buffer.rename(
            columns={"timestamp": "halter_timestamp"}
        )

        return pd.merge_asof(
            acceleration,
            reference,
            left_on="timestamp",
            right_on="halter_timestamp",
            direction="nearest",
            tolerance=MATCH_TOLERANCE,
        )


print("Chunked reading and synchronization functions ready.")

Chunked reading and synchronization functions ready.


In [12]:
def extract_window_features(frame, seconds, cow_id):
    data = frame.copy()

    data["window"] = data["timestamp"].dt.floor(f"{seconds}s")

    data["magnitude"] = np.sqrt(
        data["x"] ** 2
        + data["y"] ** 2
        + data["z"] ** 2
    )

    groups = data.groupby("window", sort=True)
    sample_counts = groups.size()

    features = pd.DataFrame(index=sample_counts.index)

    for column in ["x", "y", "z", "magnitude"]:
        signal = groups[column]

        statistics = signal.agg(
            ["mean", "std", "min", "max", "median"]
        )

        for statistic in statistics.columns:
            features[f"{column}_{statistic}"] = statistics[statistic]

        features[f"{column}_iqr"] = (
            signal.quantile(0.75) - signal.quantile(0.25)
        )

        features[f"{column}_range"] = (
            statistics["max"] - statistics["min"]
        )

        # Exact mean-square acceleration; units are mg², not joules.
        features[f"{column}_mean_square"] = (
            data[column].pow(2).groupby(data["window"]).mean()
        )

        features[f"{column}_mean_abs_change"] = (
            groups[column]
            .diff()
            .abs()
            .groupby(data["window"])
            .mean()
        )

    features["samples"] = sample_counts
    features["matched_samples"] = groups["classification"].count()

    votes = pd.crosstab(
        data["window"],
        data["classification"],
    ).reindex(
        index=features.index,
        columns=LABELS,
        fill_value=0,
    )

    features["classification"] = votes.idxmax(axis=1).astype(int)

    features["purity"] = (
        votes.max(axis=1)
        / features["matched_samples"].replace(0, np.nan)
    )

    features["cow_id"] = cow_id

    # Exclude tied labels instead of automatically favouring label 0.
    unique_majority = (
        votes.eq(votes.max(axis=1), axis=0)
        .sum(axis=1)
        .eq(1)
    )

    maximum_gap = (
        groups["timestamp"]
        .diff()
        .dt.total_seconds()
        .groupby(data["window"])
        .max()
    )

    expected_samples = seconds * SAMPLE_RATE

    keep = (
        (sample_counts >= math.ceil(
            expected_samples * MIN_SENSOR_COVERAGE
        ))
        & (sample_counts <= math.ceil(expected_samples * 1.05))
        & (
            features["matched_samples"]
            >= sample_counts * MIN_LABEL_COVERAGE
        )
        & (maximum_gap <= MAX_INTERNAL_GAP_SECONDS)
        & unique_majority
    )

    retained = features.loc[keep].dropna()

    diagnostics = {
        "candidate_windows": len(features),
        "retained_windows": len(retained),
        "tied_or_unlabelled_windows": int((~unique_majority).sum()),
    }

    return retained, diagnostics


def process_cow(cow_id, window_sizes):
    print(f"Processing cow {cow_id:02d}...", flush=True)

    acceleration_path = DATA_DIR / f"accel-{cow_id:02d}.csv"
    halter_path = DATA_DIR / f"halter-{cow_id:02d}.csv"

    acceleration_reader = read_sensor_chunks(
        acceleration_path,
        ["timestamp", "x", "y", "z"],
    )

    halter = HalterStream(
        read_sensor_chunks(
            halter_path,
            ["timestamp", "classification"],
        )
    )

    # All requested window sizes divide this block length.
    block_seconds = math.lcm(*window_sizes)

    feature_parts = {seconds: [] for seconds in window_sizes}

    diagnostics = {
        "cow_id": cow_id,
        "sensor_samples": 0,
        "matched_samples": 0,
        "absolute_offset_ms_sum": 0.0,
        "maximum_absolute_offset_ms": 0.0,
        "matched_label_counts": {str(label): 0 for label in LABELS},
        "windows": {
            str(seconds): {
                "candidate_windows": 0,
                "retained_windows": 0,
                "tied_or_unlabelled_windows": 0,
            }
            for seconds in window_sizes
        },
    }

    for acceleration in complete_time_blocks(
        acceleration_reader,
        block_seconds,
    ):
        merged = halter.match(acceleration)

        offsets = (
            merged["timestamp"] - merged["halter_timestamp"]
        ).dt.total_seconds().abs() * 1000

        diagnostics["sensor_samples"] += len(merged)
        diagnostics["matched_samples"] += int(offsets.notna().sum())
        diagnostics["absolute_offset_ms_sum"] += float(offsets.sum())

        if offsets.notna().any():
            diagnostics["maximum_absolute_offset_ms"] = max(
                diagnostics["maximum_absolute_offset_ms"],
                float(offsets.max()),
            )

        for label, count in merged["classification"].value_counts().items():
            diagnostics["matched_label_counts"][str(int(label))] += int(count)

        for seconds in window_sizes:
            features, window_diagnostics = extract_window_features(
                merged,
                seconds,
                cow_id,
            )

            feature_parts[seconds].append(features)

            for key, value in window_diagnostics.items():
                diagnostics["windows"][str(seconds)][key] += value

    diagnostics["match_fraction"] = (
        diagnostics["matched_samples"]
        / max(1, diagnostics["sensor_samples"])
    )

    diagnostics["mean_absolute_offset_ms"] = (
        diagnostics.pop("absolute_offset_ms_sum")
        / max(1, diagnostics["matched_samples"])
    )

    results = {}

    for seconds in window_sizes:
        if feature_parts[seconds]:
            features = pd.concat(feature_parts[seconds])
        else:
            features = pd.DataFrame()

        results[seconds] = features

        if not features.empty:
            diagnostics["windows"][str(seconds)]["label_counts"] = {
                str(int(label)): int(count)
                for label, count in
                features["classification"].value_counts().items()
            }

            diagnostics["windows"][str(seconds)][
                "fraction_with_purity_below_0.8"
            ] = float((features["purity"] < 0.8).mean())

            features.to_pickle(
                OUTPUT_DIR / f"features_cow{cow_id:02d}_{seconds}s.pkl"
            )

    diagnostic_path = OUTPUT_DIR / f"diagnostics_cow{cow_id:02d}.json"
    diagnostic_path.write_text(
        json.dumps(diagnostics, indent=2),
        encoding="utf-8",
    )

    print(
        f"  Timestamp matches: "
        f"{diagnostics['match_fraction']:.1%}"
    )

    for seconds, features in results.items():
        if features.empty:
            raise ValueError(
                f"Cow {cow_id:02d}: no usable {seconds}s windows. "
                f"Inspect {diagnostic_path.name}."
            )

        print(
            f"  {seconds:2d}s windows: {len(features):,}; "
            f"labels: {features['classification'].value_counts().to_dict()}"
        )

    return results


print("Feature extraction functions ready.")

Feature extraction functions ready.


In [13]:
available_cows = []

for cow_id in range(1, 19):
    acceleration_exists = (
        DATA_DIR / f"accel-{cow_id:02d}.csv"
    ).is_file()

    halter_exists = (
        DATA_DIR / f"halter-{cow_id:02d}.csv"
    ).is_file()

    if acceleration_exists != halter_exists:
        raise FileNotFoundError(
            f"Cow {cow_id:02d} needs both its accel and halter CSV."
        )

    if acceleration_exists:
        available_cows.append(cow_id)

TEST_COWS = [
    cow_id for cow_id in [4, 10, 11]
    if cow_id in available_cows
]

TRAIN_COWS = [
    cow_id for cow_id in available_cows
    if cow_id not in [4, 10, 11]
]

if len(TRAIN_COWS) < 3 or not TEST_COWS:
    raise ValueError(
        "Need at least 3 training cows and at least one test cow "
        "from 04, 10, or 11."
    )

print("Training cows:", TRAIN_COWS)
print("Reserved test cows:", TEST_COWS)

training_parts = {
    seconds: [] for seconds in WINDOW_SECONDS
}

for cow_id in TRAIN_COWS:
    cow_features = process_cow(cow_id, WINDOW_SECONDS)

    for seconds in WINDOW_SECONDS:
        training_parts[seconds].append(cow_features[seconds])

training_tables = {
    seconds: pd.concat(parts).reset_index()
    for seconds, parts in training_parts.items()
}

del training_parts
del cow_features

for seconds, table in training_tables.items():
    print(f"\n{seconds}-second training windows: {len(table):,}")
    display(
        table["classification"]
        .value_counts()
        .reindex(LABELS, fill_value=0)
        .rename(index=dict(zip(LABELS, CLASS_NAMES)))
        .to_frame("windows")
    )

Training cows: [1, 2, 3, 5]
Reserved test cows: [4]
Processing cow 01...
  Timestamp matches: 100.0%
   5s windows: 290,160; labels: {0: 117577, 1: 116473, 2: 56110}
  10s windows: 145,080; labels: {0: 58788, 1: 58235, 2: 28057}
  20s windows: 72,415; labels: {0: 29376, 1: 29061, 2: 13978}
  60s windows: 24,044; labels: {0: 9901, 1: 9498, 2: 4645}
Processing cow 02...
  Timestamp matches: 100.0%
   5s windows: 92,160; labels: {1: 37935, 0: 33562, 2: 20663}
  10s windows: 46,080; labels: {1: 18967, 0: 16780, 2: 10333}
  20s windows: 23,010; labels: {1: 9441, 0: 8404, 2: 5165}
  60s windows: 7,626; labels: {1: 3106, 0: 2836, 2: 1684}
Processing cow 03...
  Timestamp matches: 100.0%
   5s windows: 225,720; labels: {0: 94035, 1: 83937, 2: 47748}
  10s windows: 112,860; labels: {0: 47018, 1: 41971, 2: 23871}
  20s windows: 56,087; labels: {0: 23455, 1: 20923, 2: 11709}
  60s windows: 18,705; labels: {0: 7943, 1: 7011, 2: 3751}
Processing cow 05...
  Timestamp matches: 100.0%
   5s windows: 

,windows
classification,
Other,286024
Ruminating,274174
Eating_or_Drinking,139642



10-second training windows: 349,920


,windows
classification,
Other,143011
Ruminating,137089
Eating_or_Drinking,69820



20-second training windows: 174,408


,windows
classification,
Other,71443
Ruminating,68369
Eating_or_Drinking,34596



60-second training windows: 57,980


,windows
classification,
Other,24114
Ruminating,22540
Eating_or_Drinking,11326


In [14]:
MODEL_CONFIGURATIONS = [
    {
        "name": "balanced_depth12_leaf10",
        "max_depth": 12,
        "min_samples_leaf": 10,
        "class_weight": "balanced_subsample",
    },
    {
        "name": "balanced_depth20_leaf4",
        "max_depth": 20,
        "min_samples_leaf": 4,
        "class_weight": "balanced_subsample",
    },
    {
        "name": "unweighted_depth20_leaf4",
        "max_depth": 20,
        "min_samples_leaf": 4,
        "class_weight": None,
    },
]


def make_model(configuration_index):
    parameters = {
        key: value
        for key, value in MODEL_CONFIGURATIONS[configuration_index].items()
        if key != "name"
    }

    return RandomForestClassifier(
        n_estimators=N_TREES,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        **parameters,
    )


def predictor_columns(table):
    return [
        column
        for column in table.columns
        if column not in METADATA_COLUMNS + ["window"]
    ]


validation_rows = []

for seconds, table in training_tables.items():
    feature_columns = predictor_columns(table)

    X = table[feature_columns]
    y = table["classification"].astype(int)
    groups = table["cow_id"]

    if set(y.unique()) != set(LABELS):
        raise ValueError(
            f"The {seconds}s training data is missing a behavior class."
        )

    splitter = GroupKFold(
        n_splits=min(4, len(TRAIN_COWS))
    )

    folds = list(splitter.split(X, y, groups))

    for configuration_index, configuration in enumerate(
        MODEL_CONFIGURATIONS
    ):
        print(
            f"\nWindow: {seconds}s | Model: {configuration['name']}",
            flush=True,
        )

        for fold_number, (train_indices, validation_indices) in enumerate(
            folds,
            start=1,
        ):
            fold_y_train = y.iloc[train_indices]

            if set(fold_y_train.unique()) != set(LABELS):
                raise ValueError(
                    "A training fold is missing a behavior class. "
                    "More training animals or labelled data are needed."
                )

            model = make_model(configuration_index)

            model.fit(
                X.iloc[train_indices],
                fold_y_train,
            )

            training_predictions = model.predict(X.iloc[train_indices])

            training_f1 = f1_score(
                fold_y_train,
                training_predictions,
                labels=LABELS,
                average="macro",
                zero_division=0,
            )

            validation_predictions = model.predict(
                X.iloc[validation_indices]
            )

            validation_cows = groups.iloc[
                validation_indices
            ].to_numpy()

            validation_labels = y.iloc[
                validation_indices
            ].to_numpy()

            for cow_id in sorted(np.unique(validation_cows)):
                cow_mask = validation_cows == cow_id

                cow_labels = validation_labels[cow_mask]
                cow_predictions = validation_predictions[cow_mask]

                report = classification_report(
                    cow_labels,
                    cow_predictions,
                    labels=LABELS,
                    target_names=CLASS_NAMES,
                    output_dict=True,
                    zero_division=0,
                )

                row = {
                    "window_seconds": seconds,
                    "configuration_index": configuration_index,
                    "configuration": configuration["name"],
                    "fold": fold_number,
                    "validation_cow": int(cow_id),
                    "training_macro_f1": float(training_f1),
                    "validation_macro_f1": report["macro avg"]["f1-score"],
                    "validation_accuracy": accuracy_score(
                        cow_labels,
                        cow_predictions,
                    ),
                }

                for class_name in CLASS_NAMES:
                    for metric in ["precision", "recall", "f1-score"]:
                        row[f"{class_name}_{metric}"] = report[class_name][metric]

                validation_rows.append(row)

            print(
                f"  Fold {fold_number}/{len(folds)} finished; "
                f"validation cows: {sorted(np.unique(validation_cows))}",
                flush=True,
            )

        # Save progress after each candidate.
        pd.DataFrame(validation_rows).to_csv(
            OUTPUT_DIR / "validation_by_cow.csv",
            index=False,
        )

validation_details = pd.DataFrame(validation_rows)

validation_summary = (
    validation_details
    .groupby(
        ["window_seconds", "configuration_index", "configuration"]
    )
    .agg(
        validation_macro_f1=("validation_macro_f1", "mean"),
        between_cow_std=("validation_macro_f1", "std"),
        training_macro_f1=("training_macro_f1", "mean"),
        eating_precision=("Eating_or_Drinking_precision", "mean"),
        eating_recall=("Eating_or_Drinking_recall", "mean"),
        eating_f1=("Eating_or_Drinking_f1-score", "mean"),
    )
    .reset_index()
)

validation_summary["training_validation_gap"] = (
    validation_summary["training_macro_f1"]
    - validation_summary["validation_macro_f1"]
)

validation_summary = validation_summary.sort_values(
    "validation_macro_f1",
    ascending=False,
    kind="stable",
).reset_index(drop=True)

validation_summary.to_csv(
    OUTPUT_DIR / "validation_summary.csv",
    index=False,
)

display(validation_summary.round(4))


Window: 5s | Model: balanced_depth12_leaf10
  Fold 1/4 finished; validation cows: [np.int64(1)]
  Fold 2/4 finished; validation cows: [np.int64(3)]
  Fold 3/4 finished; validation cows: [np.int64(2)]
  Fold 4/4 finished; validation cows: [np.int64(5)]

Window: 5s | Model: balanced_depth20_leaf4
  Fold 1/4 finished; validation cows: [np.int64(1)]
  Fold 2/4 finished; validation cows: [np.int64(3)]
  Fold 3/4 finished; validation cows: [np.int64(2)]
  Fold 4/4 finished; validation cows: [np.int64(5)]

Window: 5s | Model: unweighted_depth20_leaf4
  Fold 1/4 finished; validation cows: [np.int64(1)]
  Fold 2/4 finished; validation cows: [np.int64(3)]
  Fold 3/4 finished; validation cows: [np.int64(2)]
  Fold 4/4 finished; validation cows: [np.int64(5)]

Window: 10s | Model: balanced_depth12_leaf10
  Fold 1/4 finished; validation cows: [np.int64(1)]
  Fold 2/4 finished; validation cows: [np.int64(3)]
  Fold 3/4 finished; validation cows: [np.int64(2)]
  Fold 4/4 finished; validation cows: [

,window_seconds,configuration_index,configuration,validation_macro_f1,between_cow_std,training_macro_f1,eating_precision,eating_recall,eating_f1,training_validation_gap
0,60,1,balanced_depth20_leaf4,0.8217,0.0212,0.9160,0.6419,0.8138,0.7147,0.0943
1,60,2,unweighted_depth20_leaf4,0.8206,0.0243,0.9293,0.6644,0.7784,0.7128,0.1086
2,60,0,balanced_depth12_leaf10,0.8184,0.0170,0.8510,0.6128,0.8478,0.7088,0.0326
3,20,1,balanced_depth20_leaf4,0.7861,0.0252,0.8869,0.5792,0.7678,0.6576,0.1008
4,20,2,unweighted_depth20_leaf4,0.7860,0.0292,0.9043,0.6105,0.7167,0.6558,0.1183
5,20,0,balanced_depth12_leaf10,0.7811,0.0207,0.8049,0.5445,0.8159,0.6507,0.0238
6,10,1,balanced_depth20_leaf4,0.7699,0.0263,0.8657,0.5531,0.7564,0.6359,0.0959
7,10,2,unweighted_depth20_leaf4,0.7695,0.0307,0.8813,0.5891,0.6928,0.6330,0.1118
8,10,0,balanced_depth12_leaf10,0.7653,0.0224,0.7842,0.5234,0.7999,0.6303,0.0189
9,5,2,unweighted_depth20_leaf4,0.7507,0.0296,0.8617,0.5731,0.6731,0.6157,0.1109


In [15]:
best_result = validation_summary.iloc[0]

BEST_WINDOW = int(best_result["window_seconds"])
BEST_CONFIGURATION = int(best_result["configuration_index"])

print("Selected window:", BEST_WINDOW, "seconds")
print("Selected model:", best_result["configuration"])
print(
    "Mean validation macro F1:",
    f"{best_result['validation_macro_f1']:.3f}",
)
print(
    "Training–validation F1 gap:",
    f"{best_result['training_validation_gap']:.3f}",
)

final_training_table = training_tables[BEST_WINDOW]
FINAL_FEATURE_COLUMNS = predictor_columns(final_training_table)

X_train = final_training_table[FINAL_FEATURE_COLUMNS]
y_train = final_training_table["classification"].astype(int)

final_model = make_model(BEST_CONFIGURATION)
final_model.fit(X_train, y_train)

model_package = {
    "model": final_model,
    "feature_columns": FINAL_FEATURE_COLUMNS,
    "window_seconds": BEST_WINDOW,
    "configuration": MODEL_CONFIGURATIONS[BEST_CONFIGURATION],
    "train_cows": TRAIN_COWS,
    "test_cows": TEST_COWS,
    "label_mapping": dict(zip(LABELS, CLASS_NAMES)),
    "preprocessing": {
        "sample_rate": SAMPLE_RATE,
        "match_tolerance_ms": MATCH_TOLERANCE.total_seconds() * 1000,
        "minimum_sensor_coverage": MIN_SENSOR_COVERAGE,
        "minimum_label_coverage": MIN_LABEL_COVERAGE,
        "maximum_internal_gap_seconds": MAX_INTERNAL_GAP_SECONDS,
    },
}

joblib.dump(
    model_package,
    OUTPUT_DIR / "cow_model.joblib",
    compress=3,
)

print("Final model trained and saved.")

Selected window: 60 seconds
Selected model: balanced_depth20_leaf4
Mean validation macro F1: 0.822
Training–validation F1 gap: 0.094
Final model trained and saved.


In [16]:
def evaluate_predictions(true_labels, predictions):
    report = classification_report(
        true_labels,
        predictions,
        labels=LABELS,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    report["accuracy"] = float(
        accuracy_score(true_labels, predictions)
    )

    report["confusion_matrix"] = confusion_matrix(
        true_labels,
        predictions,
        labels=LABELS,
    ).tolist()

    report["all_classes_supported"] = all(
        report[class_name]["support"] > 0
        for class_name in CLASS_NAMES
    )

    report["target_90_met"] = bool(
        report["all_classes_supported"]
        and report["accuracy"] >= 0.90
        and all(
            report[class_name][metric] >= 0.90
            for class_name in CLASS_NAMES
            for metric in ["precision", "recall", "f1-score"]
        )
    )

    return report


def display_evaluation(title, report):
    print(f"\n{title}")
    print(f"Accuracy: {report['accuracy']:.2%}")

    display(
        pd.DataFrame(
            {
                class_name: report[class_name]
                for class_name in CLASS_NAMES
            }
        ).T.round(4)
    )

    print("Confusion matrix: rows = actual, columns = predicted")

    display(
        pd.DataFrame(
            report["confusion_matrix"],
            index=[f"Actual {name}" for name in CLASS_NAMES],
            columns=[f"Predicted {name}" for name in CLASS_NAMES],
        )
    )

    print(
        "Accuracy and every class's precision/recall/F1 reach 90%:",
        report["target_90_met"],
    )


test_reports = {}
prediction_tables = []

for cow_id in TEST_COWS:
    test_features = process_cow(
        cow_id,
        [BEST_WINDOW],
    )[BEST_WINDOW]

    X_test = test_features[FINAL_FEATURE_COLUMNS]
    y_test = test_features["classification"].astype(int)

    predictions = final_model.predict(X_test)

    report = evaluate_predictions(y_test, predictions)
    test_reports[f"cow_{cow_id:02d}"] = report

    display_evaluation(f"TEST COW {cow_id:02d}", report)

    prediction_table = test_features[METADATA_COLUMNS].copy()
    prediction_table["prediction"] = predictions
    prediction_tables.append(prediction_table)

all_test_predictions = pd.concat(prediction_tables)

all_test_predictions.to_csv(
    OUTPUT_DIR / "test_predictions.csv"
)

pooled_report = evaluate_predictions(
    all_test_predictions["classification"],
    all_test_predictions["prediction"],
)

test_reports["pooled_test"] = pooled_report

test_reports["all_test_cows_meet_90"] = all(
    test_reports[f"cow_{cow_id:02d}"]["target_90_met"]
    for cow_id in TEST_COWS
)

test_reports["selected_window_seconds"] = BEST_WINDOW
test_reports["selected_configuration"] = (
    MODEL_CONFIGURATIONS[BEST_CONFIGURATION]["name"]
)
test_reports["training_cows"] = TRAIN_COWS
test_reports["test_cows"] = TEST_COWS

training_report = evaluate_predictions(
    y_train,
    final_model.predict(X_train),
)
test_reports["training_fit"] = training_report

(OUTPUT_DIR / "test_report.json").write_text(
    json.dumps(test_reports, indent=2),
    encoding="utf-8",
)

display_evaluation("ALL TEST COWS COMBINED", pooled_report)

print("\nResults saved to:", OUTPUT_DIR.resolve())
print("Small files you can share:")
print(" - test_report.json")
print(" - validation_summary.csv")
print(" - diagnostics_cowXX.json")

Processing cow 04...
  Timestamp matches: 100.0%
  60s windows: 8,465; labels: {0: 4016, 1: 3073, 2: 1376}

TEST COW 04
Accuracy: 83.84%


,precision,recall,f1-score,support
Other,0.8799,0.9246,0.9017,4016.0
Ruminating,0.8748,0.7869,0.8285,3073.0
Eating_or_Drinking,0.6523,0.7020,0.6762,1376.0


Confusion matrix: rows = actual, columns = predicted


,Predicted Other,Predicted Ruminating,Predicted Eating_or_Drinking
Actual Other,3713,156,147
Actual Ruminating,287,2418,368
Actual Eating_or_Drinking,220,190,966


Accuracy and every class's precision/recall/F1 reach 90%: False

ALL TEST COWS COMBINED
Accuracy: 83.84%


,precision,recall,f1-score,support
Other,0.8799,0.9246,0.9017,4016.0
Ruminating,0.8748,0.7869,0.8285,3073.0
Eating_or_Drinking,0.6523,0.7020,0.6762,1376.0


Confusion matrix: rows = actual, columns = predicted


,Predicted Other,Predicted Ruminating,Predicted Eating_or_Drinking
Actual Other,3713,156,147
Actual Ruminating,287,2418,368
Actual Eating_or_Drinking,220,190,966


Accuracy and every class's precision/recall/F1 reach 90%: False

Results saved to: C:\Users\firdaus.amyar\Downloads\SamsungSFT\backend\notebooks\cow_results_20260907_222727
Small files you can share:
 - test_report.json
 - validation_summary.csv
 - diagnostics_cowXX.json
